Download the data from the below gdrive and upload into the catalog
https://drive.google.com/drive/folders/1J3AVJIPLP7CzT15yJIpSiWXshu1iLXKn?usp=drive_link

##**1. Data Munging** -

####1. Visibily/Manually opening the file and capture couple of data patterns (Manual Exploratory Data Analysis)

### Logistic Source 1 file<br>
- Header present (shipment_id,first_name,last_name,age,role) and no Trailer<br>
- Null data, missing data for role and additional column added in between<br>
- Age is not only int, its in string type (ten in the place of '10')<br>


####2. Programatically try to find couple of data patterns applying below EDA (File: logistics_source1)
1. Apply inferSchema and toDF to create a DF and analyse the actual data.
2. Analyse the schema, datatypes, columns etc.,
3. Analyse the duplicate records count and summary of the dataframe.

In [0]:
log_src1_df1=spark.read.format("csv").options(inferSchema=True,header=True).load(path="/Volumes/enterprise_fleet_analytics_pipeline/logistic/source/Fleet Shipment Source/logistics_source1").toDF("shipment_id","first_name","last_name","age","role")
#display(log_src1_df1.printSchema())
print(log_src1_df1.printSchema())
#display(log_src1_df1)
#print(log_src1_df1.count())
log_src1_df2=log_src1_df1.dropDuplicates()
#display(log_src1_df2.orderBy("shipment_id"))
print(log_src1_df2.count())
display(log_src1_df2.summary())
display(log_src1_df2.describe())

In [0]:
log_src1_df2.createOrReplaceTempView("log_src1_tv1")
spark.sql("select * from log_src1_tv1").show()

In [0]:
%sql
select * from log_src1_tv1 limit 10;

In [0]:
%sql
create or replace temporary view log_src1_tv2 
using csv 
options (
  path "/Volumes/enterprise_fleet_analytics_pipeline/logistic/source/Fleet Shipment Source/logistics_source1",
  header "true",
  inferSchema "true"
);

--OR

create or replace temporary view log_src1_tv3(
  id int comment 'shipment_id', --comment is optional
  f_name string comment 'first_name',
  l_name string comment 'last_name',
  age int comment 'age',
  role string comment 'profession'
)
using csv
options(
  path "/Volumes/enterprise_fleet_analytics_pipeline/logistic/source/Fleet Shipment Source/logistics_source1",
  header "false",
  inferSchema "false"
);

select * from log_src1_tv3 limit 10;
describe log_src1_tv3; --by default comment is null 

###a. Passive Data Munging -  (File: logistics_source1  and logistics_source2)
Without modifying the data, identify:<br>
Shipment IDs that appear in both master_v1 and master_v2<br>
Records where:<br>
1. shipment_id is non-numeric
2. age is not an integer<br>

Count rows having:
3. fewer columns than expected
4. more columns than expected

In [0]:
#Create a Spark Session Object
from pyspark.sql import SparkSession 
sparksessionobj = SparkSession.builder.appName("logistics").getOrCreate()
src1_schema="shipment_id string, first_name string, last_name string, age string, role string"
src1_df=sparksessionobj.read.schema(src1_schema).csv(path="/Volumes/enterprise_fleet_analytics_pipeline/logistic/source/Fleet Shipment Source/",header=True,inferSchema=True,recursiveFileLookup=True,pathGlobFilter='*1')
#display(src1_df)
print(src1_df.count())
src2_schema="shipment_id string, first_name string, last_name string, age string, role string, hub_location string, vehicle_type string"
src2_df=sparksessionobj.read.schema(src2_schema).csv(path="/Volumes/enterprise_fleet_analytics_pipeline/logistic/source/Fleet Shipment Source/",header=True,inferSchema=True,recursiveFileLookup=True,pathGlobFilter='*2')
#display(src2_df)
print(src2_df.count())
final_src_df=src1_df.unionByName(src2_df,allowMissingColumns=True)
display(final_src_df)
print(final_src_df.count())

###**b. Active Data Munging** File: logistics_source1 and logistics_source2

#####1.Combining Data + Schema Merging (Structuring)
1. Read both files without enforcing schema
2. Align them into a single canonical schema: shipment_id,
first_name,
last_name,
age,
role,
hub_location,
vehicle_type,
data_source
3. Add data_source column with values as: system1, system2 in the respective dataframes

In [0]:
from pyspark.sql.functions import lit,col

src1_df1=src1_df.withColumn('data_source',lit('system1'))
#display(src1_df1)

src2_df1=src2_df.withColumn('data_source',lit('system2'))
#display(src2_df1)

final_src_df1=src1_df1.unionByName(src2_df1,allowMissingColumns=True)
final_src_df2=final_src_df1.select('shipment_id','first_name','last_name','age','role','hub_location','vehicle_type','data_source')
display(final_src_df2)

#####2. Cleansing, Scrubbing: 
Cleansing (removal of unwanted datasets)<br>
1. Mandatory Column Check - Drop any record where any of the following columns is NULL:shipment_id, role<br>
2. Name Completeness Rule - Drop records where both of the following columns are NULL: first_name, last_name<br>
3. Join Readiness Rule - Drop records where the join key is null: shipment_id<br>

Scrubbing (convert raw to tidy)<br>
4. Age Defaulting Rule - Fill NULL values in the age column with: -1<br>
5. Vehicle Type Default Rule - Fill NULL values in the vehicle_type column with: UNKNOWN<br>
6. Invalid Age Replacement - Replace the following values in age:
"ten" to -1
"" to -1<br>
7. Vehicle Type Normalization - Replace inconsistent vehicle types: 
truck to LMV
bike to TwoWheeler

In [0]:
#1. Mandatory Column Check - Drop any record where any of the following columns is NULL:shipment_id, role
#display(final_src_df2.where('shipment_id is null'))
#display(final_src_df2.where('role is null'))
#print(final_src_df2.count())
cleaned_src_df1=final_src_df2.na.drop(how='any', subset=['shipment_id','role'])
#print(cleaned_src_df1.count())
#display(cleaned_src_df1.limit(10))

#2. Name Completeness Rule - Drop records where both of the following columns are NULL: first_name, last_name
#print(cleaned_src_df1.count())
cleaned_src_df2=cleaned_src_df1.na.drop(how='all',subset=['first_name','last_name'])
#print(cleaned_src_df2.count())

#3. Join Readiness Rule - Drop records where the join key is null: shipment_id
#display(cleaned_src_df2.where('shipment_id is null'))

#4. Age Defaulting Rule - Fill NULL values in the age column with: -1
#display(cleaned_src_df2.where('age is null'))
scrubbed_src_df1=cleaned_src_df2.na.fill('-1',subset=['age'])
#display(scrubbed_src_df1.where('age = "-1"'))
print(scrubbed_src_df1.count())

#5. Vehicle Type Default Rule - Fill NULL values in the vehicle_type column with: UNKNOWN
#print("vehicle is null")
#display(scrubbed_src_df1.where('vehicle_type is null').count())
scrubbed_src_df2=scrubbed_src_df1.na.fill('UNKNOWN',['vehicle_type'])

#6. Invalid Age Replacement - Replace the following values in age: "ten" to -1 "" to -1
#display(scrubbed_src_df2.where('age = "ten"'))
age_value_updates={"ten":"-1"}
scrubbed_src_df3=scrubbed_src_df2.na.replace(age_value_updates,['age'])
#display(scrubbed_src_df3.where('age = ""'))

#7. Vehicle Type Normalization - Replace inconsistent vehicle types: truck to LMV bike to TwoWheeler
vechile_types={"Truck":"LMV",
               "Bike":"TwoWheeler"
               }
scrubbed_src_df4=scrubbed_src_df3.na.replace(vechile_types,['vehicle_type'])
#display(scrubbed_src_df4.select('*').groupBy('vehicle_type').count())
display(scrubbed_src_df4)


####3. Standardization, De-Duplication and Replacement / Deletion of Data to make it in a usable format

Detail Dataframe creation <br>
1. Read Data from logistics_shipment_detail.json
2. As this data is a clean json data, it doesn't require any cleansing or scrubbing.

In [0]:
log_shipment_df1=spark.read.json(path='/Volumes/enterprise_fleet_analytics_pipeline/logistic/source/Fleet Shipment Source/logistics_shipment_detail_3000.json',multiLine=True)
display(log_shipment_df1.sample(.1))


Standardizations:<br>

1. Add a column<br> 
Source File: DF of logistics_shipment_detail_3000.json<br>: domain as 'Logistics',  current timestamp 'ingestion_timestamp' and 'False' as 'is_expedited'
2. Column Uniformity: 
role - Convert to lowercase<br>
Source File: logistics_source1 & logistics_source2<br>
vehicle_type - Convert values to UPPERCASE<br>
Source Files: The merged master files
hub_location - Convert values to initcap case<br>
3. Format Standardization:<br>
Source Files: logistics_shipment_detail_3000.json
Convert shipment_ref to string<br>
Pad to 10 characters with leading zeros<br>
Convert dispatch_date to yyyy-MM-dd<br>
Ensure delivery_cost has 2 decimal precision<br>
4. Data Type Standardization<br>
Standardizing column data types to fix schema drift and enable mathematical operations.<br>
Source File: logistics_source1 & logistics_source2 <br>
age: Cast String to Integer<br>
Source File: logistics_shipment_detail_3000.json<br>
shipment_weight_kg: Cast to Double<br>
Source File: logistics_shipment_detail_3000.json<br>
is_expedited: Cast to Boolean<br>
5. Naming Standardization <br>
Source File: logistics_source1 & logistics_source2<br>
Rename: first_name to staff_first_name<br>
Rename: last_name to staff_last_name<br>
Rename: hub_location to origin_hub_city<br>
6. Reordering columns logically in a better standard format:<br>
Source File: All 3 files<br>
shipment_id (Identifier), staff_first_name (Dimension)staff_last_name (Dimension), role (Dimension), origin_hub_city (Location), shipment_cost (Metric), ingestion_timestamp (Audit)

In [0]:
from pyspark.sql.functions import upper,lower,initcap,lit,col,to_date,lpad,current_timestamp,date_trunc

#1. Add a column Source File: logistics_shipment_detail_3000.json: domain as 'Logistics' current timestamp 'ingestion_timestamp' and 'False' as 'is_expedited'
log_shipment_df2=log_shipment_df1.withColumns({'domain':lit('logistics'),
                                               'ingestion_timestamp':current_timestamp(),
                                               'is_expedited':lit('False')
                                               })
#display(log_shipment_df2.sample(.1))

#2. Column Uniformity: role - Convert to lowercase
#Source File: logistics_source1 & logistics_source2
#vehicle_type - Convert values to UPPERCASE
#Source Files: The merged master files hub_location - Convert values to initcap case
std_src_df1=scrubbed_src_df4.withColumns({
    'role':lower(scrubbed_src_df4.role),
    'vehicle_type':upper(scrubbed_src_df4.vehicle_type),
    'hub_location':initcap(scrubbed_src_df4.hub_location)
})
#display(std_src_df1)


#3. Format Standardization:
#Source Files: logistics_shipment_detail_3000.json Convert shipment_ref to string
#Pad to 10 characters with leading zeros
#Convert dispatch_date to yyyy-MM-dd
#Ensure delivery_cost has 2 decimal precision

#log_shipment_df2.printSchema()
log_shipment_df3=log_shipment_df2.withColumn('shipment_id',col('shipment_id').cast('string'))
#log_shipment_df3.printSchema()
#display(log_shipment_df3.sample(.1))
log_shipment_df4=log_shipment_df3.withColumn('shipment_id',lpad(col("shipment_id"),10,'0'))
log_shipment_df5=log_shipment_df4.withColumn('shipment_date',to_date(col('shipment_date'),'yyyy-MM-dd'))
#log_shipment_df5.printSchema()
log_shipment_df6=log_shipment_df5.withColumn('shipment_cost',col('shipment_cost').cast('decimal(10,2)'))
#display(log_shipment_df6)

#4. Data Type Standardization
#Standardizing column data types to fix schema drift and enable mathematical operations.
#Source File: DF of merged(logistics_source1 & logistics_source2)
#age: Cast String to Integer
#Source File: DF of logistics_shipment_detail_3000.json
#shipment_weight_kg: Cast to Double
#Source File: DF of logistics_shipment_detail_3000.json
#is_expedited: Cast to Boolean
#check for non integer value in shipment_id and age column 
#display(std_src_df1.where("shipment_id rlike '[a-zA-Z]'"))
#display(std_src_df1.where("age rlike '[^0-9]'")) #Age is having -1
std_src_df2=std_src_df1.withColumn('age',col('age').cast('int'))
log_shipment_df7=log_shipment_df6.withColumn('shipment_weight_kg',col('shipment_weight_kg').cast('double')).withColumn('is_expedited',col('is_expedited').cast('boolean'))
#log_shipment_df7.printSchema()

#5. Naming Standardization
#Source File: DF of merged(logistics_source1 & logistics_source2)
#Rename: first_name to staff_first_name
#Rename: last_name to staff_last_name
#Rename: hub_location to origin_hub_city

#std_src_df2.printSchema()
std_src_df3=std_src_df2.withColumnsRenamed({
    'first_name':'staff_first_name',
    'last_name':'staff_last_name',
    'hub_location':'origin_hub_city'
})
#display(std_src_df3.select('*'))

#6. Reordering columns logically in a better standard format:
std_src_df4=std_src_df3.withColumn('ingestion_timestamp',date_trunc("second",current_timestamp()))
#std_src_df4.printSchema()
std_final_df=std_src_df4.select("shipment_id","staff_first_name","staff_last_name","role","origin_hub_city","age","vehicle_type","ingestion_timestamp")
#display(std_final_df)
#log_shipment_df7.printSchema()
log_final_df=log_shipment_df7.select("shipment_id","order_id","source_city","destination_city","shipment_status","cargo_type","vehicle_type","payment_mode","shipment_weight_kg","shipment_cost","shipment_date","is_expedited","ingestion_timestamp")
#display(log_final_df.sample(.1))

Deduplication:
1. Apply Record Level De-Duplication
2. Apply Column Level De-Duplication (Primary Key Enforcement)

In [0]:
#1. Apply Record Level De-Duplication
#display(std_final_df.count())
#print("")
#display(std_final_df.distinct().count())
#display(std_final_df.orderBy('shipment_id'))
deduplicated_src_df=std_final_df.distinct() #row level duplicates removed
display(deduplicated_src_df.count())
print("")
#no row duplicate in log_final_df
deduplicated_log_df=log_final_df.distinct()
display(deduplicated_log_df.count())
print("")
#2. Apply Column Level De-Duplication (Primary Key Enforcement)
src_pf_df=deduplicated_src_df.dropDuplicates(['shipment_id'])
json_pf_df=deduplicated_log_df.dropDuplicates(['order_id'])
display(src_pf_df.count())
print("")
display(json_pf_df.count())

##2. Data Enrichment - Detailing of data
Makes your data rich and detailed <br>

###### Adding of Columns (Data Enrichment)
*Creating new derived attributes to enhance traceability and analytical capability.*

**1. Add Audit Timestamp (`load_dt`)**
Source File: logistics_source1 and logistics_source2<br>
* **Scenario:** We need to track exactly when this record was ingested into our Data Lakehouse for auditing purposes.
* **Action:** Add a column `load_dt` using the function `current_timestamp()`.

**2. Create Full Name (`full_name`)**
Source File: logistics_source1 and logistics_source2<br>
* **Scenario:** The reporting dashboard requires a single field for the driver's name instead of separate columns.
* **Action:** Create `full_name` by concatenating `first_name` and `last_name` with a space separator.
* **Result:** "Rajesh" + " " + "Kumar" -> **"Rajesh Kumar"**

**3. Define Route Segment (`route_segment`)**
Source File: logistics_shipment_detail_3000.json<br>
* **Scenario:** The logistics team wants to analyze performance based on specific transport lanes (Source to Destination).
* **Action:** Combine `source_city` and `destination_city` with a hyphen.
* **Result:** "Chennai" + "-" + "Pune" -> **"Chennai-Pune"**

**4. Generate Vehicle Identifier (`vehicle_identifier`)**
Source File: logistics_shipment_detail_3000.json<br>
* **Scenario:** We need a unique tracking code that immediately tells us the vehicle type and the shipment ID.
* **Action:** Combine `vehicle_type` and `shipment_id` to create a composite key.
* **Result:** "Truck" + "_" + "500001" -> **"Truck_500001"**

In [0]:
from pyspark.sql.functions import current_date,concat
#load_dt
de_src_df1=src_pf_df.withColumn('load_dt',current_date())
#display(de_src_df1)

de_src_df2=de_src_df1.withColumn('full_name',concat(col('staff_first_name'),lit(' '),col('staff_last_name')))
#display(de_src_df2) 

de_json_df1=json_pf_df.withColumn('route_segment',concat(col("source_city"),lit("-"),col("destination_city")))
#display(de_json_df1.sample(.1))

de_json_df2=de_json_df1.withColumn('vehicle_identifier',concat(col("vehicle_type"),lit("_"),col("shipment_id")))
display(de_json_df2.sample(.1))

###### Deriving of Columns (Time Intelligence)
*Extracting temporal features from dates to enable period-based analysis and reporting.*<br>
Source File: logistics_shipment_detail_3000.json<br>
**1. Derive Shipment Year (`shipment_year`)**
* **Scenario:** Management needs an annual performance report to compare growth year-over-year.
* **Action:** Extract the year component from `shipment_date`.
* **Result:** "2024-04-23" -> **2024**

**2. Derive Shipment Month (`shipment_month`)**
* **Scenario:** Analysts want to identify seasonal peaks (e.g., increased volume in December).
* **Action:** Extract the month component from `shipment_date`.
* **Result:** "2024-04-23" -> **4** (April)

**3. Flag Weekend Operations (`is_weekend`)**
* **Scenario:** The Operations team needs to track shipments handled during weekends to calculate overtime pay or analyze non-business day capacity.
* **Action:** Flag as **'True'** if the `shipment_date` falls on a Saturday or Sunday.

In [0]:
from pyspark.sql.functions import year,month,dayofweek

de_json_df3=de_json_df2.withColumns({
    'shipment_year':year(col('shipment_date')),
    'shipment_month':month(col('shipment_date'))
})

de_json_df4=de_json_df3.withColumn('is_weekend',dayofweek(col('shipment_date')).isin(1,7))
display(de_json_df4.sample(.1))

###### Enrichment/Business Logics (Calculated Fields)
*Deriving new metrics and financial indicators using mathematical and date-based operations.*<br>
Source File: logistics_shipment_detail_3000.json<br>

**1. Calculate Unit Cost (`cost_per_kg`)**
* **Scenario:** The Finance team wants to analyze the efficiency of shipments by determining the cost incurred per unit of weight.
* **Action:** Divide `shipment_cost` by `shipment_weight_kg`.
* **Logic:** `shipment_cost / shipment_weight_kg`

**2. Track Shipment Age (`days_since_shipment`)**
* **Scenario:** The Operations team needs to monitor how long it has been since a shipment was dispatched to identify potential delays.
* **Action:** Calculate the difference in days between the `current_date` and the `shipment_date`.
* **Logic:** `datediff(current_date(), shipment_date)`

**3. Compute Tax Liability (`tax_amount`)**
* **Scenario:** For invoicing and compliance, we must calculate the Goods and Services Tax (GST) applicable to each shipment.
* **Action:** Calculate 18% GST on the total `shipment_cost`.
* **Logic:** `shipment_cost * 0.18`

In [0]:
from pyspark.sql.functions import round,bround,date_diff,current_date

de_json_df5=de_json_df4.withColumn('cost_per_kg',bround(col('shipment_cost')/col('shipment_weight_kg'),2)).withColumn('days_since_shipment',date_diff(current_date(), col('shipment_date')))
#display(de_json_df5.sample(.1))
gst=de_json_df5.shipment_cost * .18
de_json_df6=de_json_df5.withColumn('gst',round(gst,2))
display(de_json_df6.sample(.1))

###### Remove/Eliminate (drop, select, selectExpr)
*Excluding unnecessary or redundant columns to optimize storage and privacy.*<br>
Source File: logistics_source1 and logistics_source2<br>

**1. Remove Redundant Name Columns**
* **Scenario:** Since we have already created the `full_name` column in the Enrichment step, the individual name columns are now redundant and clutter the dataset.
* **Action:** Drop the `first_name` and `last_name` columns.
* **Logic:** `df.drop("first_name", "last_name")`

In [0]:
de_src_df3=de_src_df2.drop('staff_first_name','staff_last_name')
de_src_df4=de_src_df3.select("shipment_id","full_name","age","role","origin_hub_city","vehicle_type","ingestion_timestamp","load_dt")
display(de_src_df4)

##### Splitting & Merging/Melting of Columns
*Reshaping columns to extract hidden values or combine fields for better analysis.*<br>
Source File: logistics_shipment_detail_3000.json<br>
**1. Splitting (Extraction)**
*Breaking one column into multiple to isolate key information.*
* **Split Order Code:**
  * **Action:** Split `order_id` ("ORD100000") into two new columns:
    * `order_prefix` ("ORD")
    * `order_sequence` ("100000")
* **Split Date:**
  * **Action:** Split `shipment_date` into three separate columns for partitioning:
    * `ship_year` (2024)
    * `ship_month` (4)
    * `ship_day` (23)

**2. Merging (Concatenation)**
*Combining multiple columns into a single unique identifier or description.*
* **Create Route ID:**
  * **Action:** Merge `source_city` ("Chennai") and `destination_city` ("Pune") to create a descriptive route key:
    * `route_lane` ("Chennai->Pune")

In [0]:
from pyspark.sql.functions import regexp_extract,day

de_json_df7=de_json_df6.withColumns({
    'order_prefix':regexp_extract(col('order_id'),'^([A-Za-z]+)',1),
    'order_sequence':regexp_extract(col('order_id'),'([0-9]+)$',1)
})


de_json_df8=de_json_df7.withColumns({
    'ship_date':day(col('shipment_date'))
}).withColumnsRenamed({
    'shipment_year':'ship_year',
    'shipment_month':'ship_month'
})
display(de_json_df8.limit(10))

## 3. Data Customization & Processing - Application of Tailored Business Specific Rules

### **UDF1: Complex Incentive Calculation**
**Scenario:** The Logistics Head wants to calculate a "Performance Bonus" for drivers based on tenure and role complexity.

**Action:** Create a Python function `calculate_bonus(role, age)` and register it as a Spark UDF.

**Logic:**
* **IF** `Role` == 'Driver' **AND** `Age` > 50:
  * `Bonus` = 15% of Salary (Reward for Seniority)
* **IF** `Role` == 'Driver' **AND** `Age` < 30:
  * `Bonus` = 5% of Salary (Encouragement for Juniors)
* **ELSE**:
  * `Bonus` = 0

**Result:** A new derived column `projected_bonus` is generated for every row in the dataset.

---

### **UDF2: PII Masking (Privacy Compliance)**
**Scenario:** For the analytics dashboard, we must hide the full identity of the staff to comply with privacy laws (GDPR/DPDP), while keeping names recognizable for internal managers.

**Business Rule:** Show the first 2 letters, mask the middle characters with `****`, and show the last letter.

**Action:** Create a UDF `mask_identity(name)`.

**Example:**
* **Input:** `"Rajesh"`
* **Output:** `"Ra****h"`
<br>
**Note: Convert the above udf logic to inbult function based transformation to ensure the performance is improved.**

In [0]:
#UDF1
def calculate_bonus(role, age):
    if role=="driver":
        if age>=50:
            bonus='15%'
        elif age>30 and age<50:
            bonus='10%'
        elif age<30:
            bonus='5%'
        elif age is None:
            bonus='0%'
    else:
        bonus='0%'
    return bonus

from pyspark.sql.functions import udf,when,col
#bonus_calculation=udf(calculate_bonus)
#de_src_df5=de_src_df4.withColumn('special_bonus',bonus_calculation(col('role'),col('age')))
#display(de_src_df5)

#without using UDF
de_src_df5=de_src_df4.select('*',when((col('role')=='driver') & (col('age')>=50),'15%')
                             .when((col('role')=='driver') & ((col('age')>30) | (col('age')<50)),'10%')
                             .when((col('role')=='driver') & (col('age')<30),'5%')
                             .when((col('role')=='driver') & (col('age') is None),'0%')
                             .when(col('role')!='driver','0%').alias('special_bonus')
)
display(de_src_df5)

In [0]:
#UDF2
#python user defined function (UDF)
def full_name_masking(name):
    if name is None:
        return None
    parts=name.split()
    masked_part=[]
    for p in parts:
        if len(p)>2:
            masked_part.append(p[0]+p[1] + '*'*(len(p)-3) + p[-2]+p[-1])
        else:
            masked_part.append(p[0]+p[1] + '*')
    return ' '.join(masked_part)

#print(full_name_masking('vivekananda bharathi'))

from pyspark.sql.functions import udf
full_name_pii=udf(full_name_masking)
dp_src_df1=de_src_df4.withColumn('full_name',full_name_pii(col('full_name')))
display(dp_src_df1)

## 4. Data Core Curation & Processing (Pre-Wrangling)
*Applying business logic to focus, filter, and summarize data before final analysis.*

**1. Select (Projection)**<br>
Source Files: logistics_source1 and logistics_source2<br>
* **Scenario:** The Driver App team only needs location data, not sensitive HR info.
* **Action:** Select only `first_name`, `role`, and `hub_location`.

**2. Filter (Selection)**<br>
Source File: json<br>
* **Scenario:** We need a report on active operational problems.
* **Action:** Filter rows where `shipment_status` is **'DELAYED'** or **'RETURNED'**.
* **Scenario:** Insurance audit for senior staff.
* **Action:** Filter rows where `age > 50`.

**3. Derive Flags & Columns (Business Logic)**<br>
Source File: json<br>
* **Scenario:** Identify high-value shipments for security tracking.
* **Action:** Create flag `is_high_value` = **True** if `shipment_cost > 50,000`.
* **Scenario:** Flag weekend operations for overtime calculation.
* **Action:** Create flag `is_weekend` = **True** if day is Saturday or Sunday.

**4. Format (Standardization)**<br>
Source File: json<br>
* **Scenario:** Finance requires readable currency formats.
* **Action:** Format `shipment_cost` to string like **"₹30,695.80"**.
* **Scenario:** Standardize city names for reporting.
* **Action:** Format `source_city` to Uppercase (e.g., "chennai" → **"CHENNAI"**).

**5. Group & Aggregate (Summarization)**<br>
Source Files: logistics_source1 and logistics_source2<br>
* **Scenario:** Regional staffing analysis.
* **Action:** Group by `hub_location` and **Count** the number of staff.
* **Scenario:** Fleet capacity analysis.
* **Action:** Group by `vehicle_type` and **Sum** the `shipment_weight_kg`.

**6. Sorting (Ordering)**<br>
Source File: json<br>
* **Scenario:** Prioritize the most expensive shipments.
* **Action:** Sort by `shipment_cost` in **Descending** order.
* **Scenario:** Organize daily dispatch schedule.
* **Action:** Sort by `shipment_date` (Ascending) then `priority_flag` (Descending).

**7. Limit (Top-N Analysis)**<br>
Source File: json<br>
* **Scenario:** Dashboard snapshot of critical delays.
* **Action:** Filter for 'DELAYED', Sort by Cost, and **Limit to top 10** rows.

In [0]:
from pyspark.sql.functions import format_number, count, round, sum, regexp_replace
#1.
#display(dp_src_df1.select('full_name','role','origin_hub_city')) #.na.drop()

#2.
#display(de_json_df8.filter((col('shipment_status')=='DELAYED')|(col("shipment_status")=='RETURNED')))
#display(dp_src_df1.filter(col('age')>50))

#3.
#de_json_df9=de_json_df8.withColumn('is_high_value',col("shipment_cost")>50000.00)
#or
de_json_df9=de_json_df8.withColumn('is_high_value',when(col("shipment_cost")>20000.00,True).otherwise(False))
#display(de_json_df9.filter(col("is_high_value")=='false'))

#4.
de_json_df10=de_json_df9.withColumn('shipment_cost',col('shipment_cost').cast('double')).withColumn('shipment_cost',concat(lit('₹'),format_number(col('shipment_cost'), 2))).withColumn('source_city',upper(col('source_city')))
#display(de_json_df10)

#5
#display(dp_src_df1.groupBy('origin_hub_city').agg(count('full_name').alias('total_staff')))
#de_json_df9.printSchema()
#display(de_json_df10.groupBy("vehicle_type").agg(round(sum("shipment_weight_kg"),2).alias("total_weight")))

#6
de_json_df11=de_json_df10.select('*').orderBy(col("shipment_cost").desc())
#display(de_json_df11)
de_json_df12=de_json_df11.select('*').orderBy(col("shipment_date")).orderBy(col('is_high_value').desc())
#display(de_json_df12)

#7
de_json_df13=de_json_df12.withColumn('shipment_cost',regexp_replace(col('shipment_cost'),'₹','')).withColumn('shipment_cost',regexp_replace(col('shipment_cost'),',','').cast('double'))
display(de_json_df13.filter(col('shipment_status')=='DELAYED').orderBy(col('shipment_cost').desc()).limit(10))

## 5. Data Wrangling - Transformation & Analytics
*Combining, modeling, and analyzing data to answer complex business questions.*

### **1. Joins**
Source Files:<br>
Left Side (staff_df):<br> logistics_source1 & logistics_source2<br>
Right Side (shipments_df):<br> logistics_shipment_detail_3000.json<br>
#### **1.1 Frequently Used Simple Joins (Inner, Left)**
* **Inner Join (Performance Analysis):**
  * **Scenario:** We only want to analyze *completed work*. Connect Staff to the Shipments they handled.
  * **Action:** Join `staff_df` and `shipments_df` on `shipment_id`.
  * **Result:** Returns only rows where a staff member is assigned to a valid shipment.
* **Left Join (Idle Resource check):**
  * **Scenario:** Find out which staff members are currently *idle* (not assigned to any shipment).
  * **Action:** Join `staff_df` (Left) with `shipments_df` (Right) on `shipment_id`. Filter where `shipments_df.shipment_id` is NULL.



In [0]:
from pyspark.sql.functions import lpad
#left_df=dp_src_df1
#right_df=de_json_df13
staff_df=dp_src_df1.withColumn('shipment_id',lpad(col('shipment_id'),10,'0'))
shipments_df=de_json_df13
#display(staff_df.where(col('shipment_id')=='0005010055'))
#display(shipments_df.where(col('shipment_id')=='0005010055'))
pf_inner_df=staff_df.alias('s').join(shipments_df.alias('sh'),how='inner',on='shipment_id')
#display(pf_inner_df.filter(col('shipment_id').like('0005010055')))

irc_left_df=staff_df.alias("s").join(shipments_df.alias("sh"),col('s.shipment_id')==col('sh.shipment_id'),how="left").filter(col('sh.shipment_id').isNull())
display(irc_left_df.select("s.shipment_id","s.full_name","s.role","s.vehicle_type"))


#### **1.2 Infrequent Simple Joins (Self, Right, Full, Cartesian)**
* **Self Join (Peer Finding):**
  * **Scenario:** Find all pairs of employees working in the same `hub_location`.
  * **Action:** Join `staff_df` to itself on `hub_location`, filtering where `staff_id_A != staff_id_B`.
* **Right Join (Orphan Data Check):**
  * **Scenario:** Identify shipments in the system that have *no valid driver* assigned (Data Integrity Issue).
  * **Action:** Join `staff_df` (Left) with `shipments_df` (Right). Focus on NULLs on the left side.
* **Full Outer Join (Reconciliation):**
  * **Scenario:** A complete audit to find *both* idle drivers AND unassigned shipments in one view.
  * **Action:** Perform a Full Outer Join on `shipment_id`.
* **Cartesian/Cross Join (Capacity Planning):**
  * **Scenario:** Generate a schedule of *every possible* driver assignment to *every* pending shipment to run an optimization algorithm.
  * **Action:** Cross Join `drivers_df` and `pending_shipments_df`.

In [0]:
#display(staff_df.groupBy('origin_hub_city').count().alias('No. of Staff'))
#display(staff_df) there is no self, use inner for self join
pf_self_df=staff_df.alias("A").join(staff_df.alias("B"),how="inner",on=(col('A.origin_hub_city')==col('B.origin_hub_city'))).filter(col('A.shipment_id')!=(col('B.shipment_id')))
#display(pf_self_df)

#display(staff_df)
#display(staff_df.where(col('shipment_id')=='0005010033'))
#display(shipments_df.where(col('shipment_id')=='0005010033'))
odc_right_df=staff_df.alias("s").join(shipments_df.alias("sh"),how='right',on=(col('s.shipment_id')==col('sh.shipment_id')))
#display(odc_right_df.filter(col('s.shipment_id').isNull()))

#
rec_outer_df=staff_df.alias("s").join(shipments_df.alias("sh"),how='outer',on=(col('s.shipment_id')==col('sh.shipment_id')))
#display(rec_outer_df)

cross_join_df=staff_df.alias("s").join(shipments_df.alias("sh"))#costly join, without on condition is cross/cartesian join
display(cross_join_df)

#### **1.3 Advanced Joins (Semi and Anti)**
* **Left Semi Join (Existence Check):**
  * **Scenario:** "Show me the details of Drivers who have *at least one* shipment." (Standard filtering).
  * **Action:** `staff_df.join(shipments_df, "shipment_id", "left_semi")`.
  * **Benefit:** Performance optimization; it stops scanning the right table once a match is found.
* **Left Anti Join (Negation Check):**
  * **Scenario:** "Show me the details of Drivers who have *never* touched a shipment."
  * **Action:** `staff_df.join(shipments_df, "shipment_id", "left_anti")`.

In [0]:
semi_left_df=staff_df.alias("s").join(shipments_df.alias("sh"),how='semi',on=(col('s.shipment_id')==col('sh.shipment_id'))).filter(col('s.role')=='driver')
display(semi_left_df)

anti_left_df=staff_df.alias("s").join(shipments_df.alias("sh"),how='anti',on=(col('s.shipment_id')==col('sh.shipment_id'))).filter(col('s.role')=='driver')
display(anti_left_df)
#total driver is 17 in staff_df (5 driver's having shipment + 12 driver's dont have shipment)

### **2. Lookup**<br>
Source File: logistics_source1 and logistics_source2 (merged into Staff DF)<br>
* **Scenario:** Validation. Check if the `hub_location` in the staff file exists in the corporate `Master_City_List`.
* **Action:** Compare values against a reference list.

In [0]:
master_city_list=spark.read.csv(path='/Volumes/enterprise_fleet_analytics_pipeline/logistic/source/Fleet Shipment Source/Master_City_List.csv',header=True,inferSchema=True)
#display(master_city_list)

lookup_df=staff_df.alias("s").join(master_city_list.alias("m"),how='inner',on=(col('s.origin_hub_city')==col('m.city_name'))).selectExpr('s.shipment_id','s.full_name','s.age','s.role','m.city_name AS city_from_master')
display(lookup_df)

### **3. Lookup & Enrichment**<br>
Source File: logistics_source1 and logistics_source2 (merged into Staff DF)<br>
* **Scenario:** Geo-Tagging.
* **Action:** Lookup `hub_location` ("Pune") in a Master Latitude/Longitude table and enrich the dataset by adding `lat` and `long` columns for map plotting.

In [0]:
enrich_lookup_df=staff_df.alias("s").join(master_city_list.alias("m"),how='inner',on=(col('s.origin_hub_city')==col('m.city_name'))).selectExpr('s.shipment_id','s.full_name','s.age','s.role','m.city_name AS city_from_master','m.latitude','m.longitude')
display(enrich_lookup_df.filter(col('m.city_name')=='Pune'))

### **4. Schema Modeling (Denormalization)**<br>
Source Files: All 3 Files (logistics_source1, logistics_source2, logistics_shipment_detail_3000.json)<br>
* **Scenario:** Creating a "Gold Layer" Table for PowerBI/Tableau.
* **Action:** Flatten the Star Schema. Join `Staff`, `Shipments`, and `Vehicle_Master` into one wide table (`wide_shipment_history`) so analysts don't have to perform joins during reporting.

In [0]:
dim_staff_location_df=staff_df.alias("s").join(master_city_list.alias("m"),how="inner",on=(col('s.origin_hub_city')==col('m.city_name'))).select('shipment_id','full_name','age','role','origin_hub_city','country','ingestion_timestamp','load_dt')
#display(dim_staff_location_df)
fact_shipments=dim_staff_location_df.join(shipments_df,how="inner",on='shipment_id')
display(fact_shipments)

### **5. Windowing (Ranking & Trends)**<br>
Source Files:<br>
logistics_source2: Provides hub_location (Partition Key).<br>
logistics_shipment_detail_3000.json: Provides shipment_cost (Ordering Key)<br>
* **Scenario:** "Who are the Top 3 Drivers by Cost in *each* Hub?"
* **Action:**
  1. Partition by `hub_location`.
  2. Order by `total_shipment_cost` Descending.
  3. Apply `dense_rank()` and `row_number()
  4. Filter where `rank or row_number <= 3`.

In [0]:
from pyspark.sql.functions import row_number, desc, dense_rank
from pyspark.sql.window import Window

staff_shipments_df=staff_df.join(shipments_df,how="inner",on='shipment_id')
#display(staff_shipments_df.limit(5))
rank_df=staff_shipments_df.withColumn('rank',row_number().over(Window.partitionBy('role','origin_hub_city').orderBy(desc('shipment_cost'))))
display(rank_df.select('shipment_id','full_name','role','origin_hub_city','shipment_cost','rank')) # & (col("role")=='driver') / .filter(col("rank")<=3)

dense_rank_df=rank_df.withColumn('dense_rank',dense_rank().over(Window.partitionBy('role','origin_hub_city').orderBy(desc('shipment_cost'))))
display(dense_rank_df.select('shipment_id','full_name','role','origin_hub_city','shipment_cost','rank','dense_rank'))


### **6. Analytical Functions (Lead/Lag)**<br>
Source File: <br>
logistics_shipment_detail_3000.json<br>
* **Scenario:** Idle Time Analysis.
* **Action:** For each driver, calculate the days elapsed since their *previous* shipment.

In [0]:
from pyspark.sql.functions import lag, lead, date_diff
#display(staff_shipments_df.filter(col('role')=='driver'))
lag_shipment_date_df=staff_shipments_df.withColumn('previous_shipment_date',lag('shipment_date').over(Window.partitionBy('role').orderBy('shipment_date')))
display(lag_shipment_date_df.select('role','shipment_date','previous_shipment_date').filter(col("role")=='driver'))

days_lag_df=lag_shipment_date_df.withColumn('days_elapsed',date_diff('shipment_date','previous_shipment_date'))
display(days_lag_df.select('role','shipment_date','previous_shipment_date','days_elapsed').filter(col("role")=='driver'))

### **7. Set Operations**<br>
Source Files: logistics_source1 and logistics_source2<br>
* **Union:** Combining `Source1` (Legacy) and `Source2` (Modern) into one dataset (Already done in Active Munging).
* **Intersect:** Identifying Staff IDs that appear in *both* Source 1 and Source 2 (Duplicate/Migration Check).
* **Except (Difference):** Identifying Staff IDs present in Source 2 but *missing* from Source 1 (New Hires).

In [0]:
#display(src1_df.limit(10))
#display(src2_df.limit(10))

#union - applicable if columns are same
#unionByName - applicable if columns are different, use allowMissingColumns=True
final_src_dfnew=src1_df.unionByName(src2_df,allowMissingColumns=True)
#display(final_src_dfnew)

#intersect - support only with the same number of columns
#return common data between both df's
src2_df1=src2_df.select('shipment_id','first_name','last_name','age','role')
itr_df=src1_df.intersectAll(src2_df1)
display(itr_df)

#Difference - support only with the same number of columns
display(src1_df.count())
print('')
display(src2_df1.count())
diff_df=src2_df1.subtract(src1_df)
display(diff_df)

### **8. Grouping & Aggregations (Advanced)**<br>
Source Files:<br>
logistics_source2: Provides hub_location and vehicle_type (Grouping Dimensions).<br>
logistics_shipment_detail_3000.json: Provides shipment_cost (Aggregation Metric).<br>
* **Scenario:** The CFO wants a subtotal report at multiple levels:
  1. Total Cost by Hub.
  2. Total Cost by Hub AND Vehicle Type.
  3. Grand Total.
* **Action:** Use `cube("hub_location", "vehicle_type")` or `rollup()` to generate all these subtotals in a single query.

In [0]:
from pyspark.sql.functions import col, sum

rollup_df = fact_shipments.rollup('origin_hub_city').agg(sum(col('shipment_cost')).alias('total_shipment_cost')).orderBy('origin_hub_city', 'total_shipment_cost')
display(rollup_df)

cube_df = fact_shipments.cube('origin_hub_city','vehicle_type').agg(sum(col('shipment_cost')).alias('total_shipment_cost')).orderBy('origin_hub_city', 'total_shipment_cost')
display(cube_df)

pivot_df = fact_shipments.groupBy('origin_hub_city').pivot('vehicle_type').agg(sum(col('shipment_cost')).alias('total_shipment_cost')).orderBy('origin_hub_city')
display(pivot_df)

##6. Data Persistance (LOAD)-> Data Publishing & Consumption<br>

Store the inner joined, lookup and enrichment, Schema Modeling, windowing, analytical functions, set operations, grouping and aggregation data into the delta tables.

In [0]:
#write inner join into json
pf_inner_df1=pf_inner_df.drop(col('sh.ingestion_timestamp'),col('sh.vehicle_type')) #removed duplicate columns using alias
#display(pf_inner_df1)
pf_inner_df1.write.json("/Volumes/enterprise_fleet_analytics_pipeline/logistic/destination/inner_json",mode="overwrite")

#write enrich lookup result into delta table
enrich_lookup_df.write.saveAsTable("enterprise_fleet_analytics_pipeline_enrich_lookup",mode="overwrite")

#write into delta format
diff_df.write.format('delta').save("/Volumes/enterprise_fleet_analytics_pipeline/logistic/destination/diff_df",mode="overwrite")

# Ensure schema exists before saving rollup table
spark.sql("CREATE SCHEMA IF NOT EXISTS enterprise_fleet_analytics_pipeline.destination_schema")

# Write rollup to delta table with fully qualified name
rollup_df.write.format('delta').mode('overwrite').saveAsTable('enterprise_fleet_analytics_pipeline.destination_schema.rollup_table')

spark.sql("select * from enterprise_fleet_analytics_pipeline.destination_schema.rollup_table").show()

##7.Take the copy of the above notebook and try to write the equivalent SQL for which ever applicable.